#  介紹

本程式使用 pyautogui 來完成一個 Blitz 的遊戲機器人。

In [23]:
import pyautogui as pyg
pyg.sleep(2)
basex = 337
basey = 404
gap = 40


In [24]:
# 逐列讀取 8x8 盤面中 64 顆寶石中心點的 RGB 顏色
for row in range(8):
    colors = [
        pyg.pixel(basex + col * gap, basey + row * gap)
        for col in range(8)
    ]
    print(colors)

[(81, 253, 132), (22, 119, 238), (253, 253, 253), (184, 7, 184), (253, 253, 121), (184, 7, 184), (253, 253, 39), (253, 30, 61)]
[(81, 253, 132), (253, 30, 61), (81, 253, 132), (184, 7, 184), (253, 30, 61), (184, 7, 184), (22, 119, 238), (253, 253, 39)]
[(253, 253, 121), (253, 253, 121), (22, 119, 238), (81, 253, 132), (253, 253, 39), (253, 253, 39), (253, 253, 253), (253, 253, 39)]
[(22, 119, 238), (81, 253, 132), (253, 253, 253), (253, 253, 39), (184, 7, 184), (81, 253, 132), (253, 30, 61), (253, 30, 61)]
[(22, 119, 238), (253, 253, 39), (253, 253, 121), (22, 119, 238), (253, 253, 39), (184, 7, 184), (253, 253, 121), (253, 30, 61)]
[(184, 7, 184), (22, 119, 238), (253, 253, 39), (184, 7, 184), (253, 30, 61), (253, 253, 39), (253, 253, 39), (253, 253, 253)]
[(22, 119, 238), (81, 253, 132), (253, 253, 121), (253, 253, 39), (184, 7, 184), (253, 253, 39), (253, 30, 61), (81, 253, 132)]
[(253, 30, 61), (22, 119, 238), (184, 7, 184), (253, 30, 61), (81, 253, 132), (22, 119, 238), (253, 253,

In [25]:
def gem_position(r, c):
    x = basex + c * gap
    y = basey + r * gap
    return (x, y)

In [26]:
def get_gem_color(r, c, im):
    position = gem_position(r, c)
    return im.getpixel(position)

In [27]:
print(gem_position(0,0))
print(gem_position(0,1))
print(gem_position(1,0))

(337, 404)
(377, 404)
(337, 444)


In [28]:
im = pyg.screenshot()
for i in range(8):
    x, y = gem_position(0,i)
    pyg.click(x, y, duration=1)
    print(get_gem_color(0, i, im))

(81, 253, 132)
(22, 119, 238)
(253, 253, 253)
(184, 7, 184)
(253, 253, 121)
(184, 7, 184)
(253, 253, 39)
(253, 30, 61)


In [29]:
def get_gem_color(r, c, im):
    position = gem_position(r, c)
    return im.getpixel(position)

In [30]:
def can_move_left(r, c, im):
    # 最左欄沒有可交換的寶石，且座標必須位於 8x8 棋盤內
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    left = (r, c - 1)
    current = (r, c)
    left_color = get_gem_color(*left, im)
    current_color = get_gem_color(*current, im)
    if left_color == current_color:
        return False

    # 模擬交換後讀取顏色，不改動原始截圖。
    def color_after_swap(row, col):
        if (row, col) == left:
            return current_color
        if (row, col) == current:
            return left_color
        return get_gem_color(row, col, im)

    def forms_match(row, col):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr, nc = row + dr * direction, col + dc * direction
                while (0 <= nr < 8 and 0 <= nc < 8
                       and color_after_swap(nr, nc) == color):
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True
        return False

    return forms_match(*left) or forms_match(*current)

In [54]:
im = pyg.screenshot()
print(can_move_left(6,2,im))

False


In [ ]:
def _can_move_direction(r, c, im, dr, dc):
    neighbor = (r + dr, c + dc)
    current = (r, c)
    if not (0 <= r < 8 and 0 <= c < 8
            and 0 <= neighbor[0] < 8 and 0 <= neighbor[1] < 8):
        return False

    neighbor_color = get_gem_color(*neighbor, im)
    current_color = get_gem_color(*current, im)
    if neighbor_color == current_color:
        return False

    def color_after_swap(row, col):
        if (row, col) == neighbor:
            return current_color
        if (row, col) == current:
            return neighbor_color
        return get_gem_color(row, col, im)

    def forms_match(row, col):
        color = color_after_swap(row, col)
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr = row + axis_r * direction
                nc = col + axis_c * direction
                while (0 <= nr < 8 and 0 <= nc < 8
                       and color_after_swap(nr, nc) == color):
                    count += 1
                    nr += axis_r * direction
                    nc += axis_c * direction
            if count >= 3:
                return True
        return False

    return forms_match(*neighbor) or forms_match(*current)


def can_move_right(r, c, im):
    return _can_move_direction(r, c, im, 0, 1)


def can_move_up(r, c, im):
    return _can_move_direction(r, c, im, -1, 0)


def can_move_down(r, c, im):
    return _can_move_direction(r, c, im, 1, 0)

In [ ]:
import time

start_time = time.time()
im = pyg.screenshot()

while True:
    if time.time() - start_time > 60:
        break

    moved = False
    for r in range(8):
        for c in range(8):
            if time.time() - start_time > 60:
                break

            direction = None
            if can_move_left(r, c, im):
                direction = (0, -1)
            elif can_move_right(r, c, im):
                direction = (0, 1)
            elif can_move_up(r, c, im):
                direction = (-1, 0)
            elif can_move_down(r, c, im):
                direction = (1, 0)

            if direction is not None:
                x, y = gem_position(r, c)
                dr, dc = direction
                pyg.moveTo(x, y)
                pyg.dragTo(x + dc * gap, y + dr * gap, duration=0.15)
                moved = True
                break

        if moved or time.time() - start_time > 60:
            break

    if time.time() - start_time > 60:
        break
    im = pyg.screenshot()
    if not moved:
        pyg.sleep(0.05)